1. Importar librerías

In [ ]:
# ==========================================
# 1. Importar librerías
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)


2. Cargar dataset limpio

In [ ]:
# ==========================================
# 2. Cargar dataset limpio
# ==========================================
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
file_path = os.path.join(project_root, "data/processed/online_news_cleaned.csv")

df = pd.read_csv(file_path)
df.head()



Tabla en Markdown explicando cada variable

| Variable | Qué significa | Tipo de medida |
|---------|----------------|----------------|
| global_subjectivity | Grado de subjetividad del contenido completo | [0,1] alto=subjetivo |
| global_sentiment_polarity | Polaridad del sentimiento del texto completo | -1=negativo, +1=positivo |
| global_rate_positive_words | Proporción de palabras positivas | razón |
| global_rate_negative_words | Proporción de palabras negativas | razón |
| avg_positive_polarity | Intensidad promedio de palabras positivas | 0–1 |
| avg_negative_polarity | Intensidad promedio de palabras negativas | -1–0 |
| title_subjectivity | Subjetividad del título | 0–1 |
| title_sentiment_polarity | Polaridad del título | -1 a 1 |
| abs_title_subjectivity | Valor absoluto → cuánta subjetividad hay | 0–1 |
| abs_title_sentiment_polarity | Valor absoluto → cuánta emoción hay | 0–1 |


Interpretación para el modelo (qué aporta cada variable)

| Variable | Interpretación para el modelo |
|---------|-------------------------------|
| global_subjectivity | Contenidos más subjetivos pueden conectar emocionalmente → ligera mejora en shares |
| global_sentiment_polarity | Polaridad positiva generalmente se asocia a mejor desempeño |
| global_rate_positive_words | Mayor proporción de palabras positivas → ligera relación con viralidad |
| global_rate_negative_words | Contenido negativo puede producir picos de viralidad (contenido emocional) |
| avg_positive_polarity | Intensidad del sentimiento positivo → útil como refuerzo |
| avg_negative_polarity | Intensidad negativa → puede correlacionar con viralidad extrema |
| title_subjectivity | Títulos subjetivos atraen más clics |
| title_sentiment_polarity | Títulos positivos generan más engagement |
| abs_title_subjectivity | Subjetividad extrema indica títulos "clickbait" → potencial de viralidad |
| abs_title_sentiment_polarity | Polaridad extrema refuerza viralidad → títulos muy emocionales |


📊 Tabla — Estadísticas descriptivas del Grupo E

In [ ]:
sentiment_vars = [
    'global_subjectivity','global_sentiment_polarity',
    'global_rate_positive_words','global_rate_negative_words',
    'avg_positive_polarity','avg_negative_polarity',
    'title_subjectivity','title_sentiment_polarity',
    'abs_title_subjectivity','abs_title_sentiment_polarity'
]

df[sentiment_vars].describe().T


📈 5. Histogramas + KDE de cada variable

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 18))
axes = axes.flatten()

for ax, col in zip(axes, sentiment_vars):
    sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Distribución de {col}")

plt.tight_layout()
plt.show()


6. Correlación entre sentimiento y viralidad (shares)

In [ ]:
corr_sent = df[sentiment_vars + ['shares']].corr()['shares'].sort_values(ascending=False)
corr_sent


7. Dispersión con regresión (sentimiento vs shares)

In [ ]:
for col in sentiment_vars:
    sns.lmplot(data=df, x=col, y='shares',
               scatter_kws={'alpha':0.2}, line_kws={'color':'red'})
    plt.title(f"Relación entre {col} y shares")


8. log(shares) vs sentimiento (log–log cuando aplica)

In [ ]:
df['log_shares'] = np.log1p(df['shares'])

for col in ['global_sentiment_polarity', 'avg_positive_polarity', 'avg_negative_polarity']:
    plt.figure(figsize=(8,5))
    sns.scatterplot(x=df[col], y=df['log_shares'], alpha=0.2)
    plt.xlabel(col)
    plt.ylabel("log(shares)")
    plt.title(f"log(shares) vs {col}")
    plt.show()


🟦 9. Comparación entre artículos positivos / negativos / neutros

In [ ]:
df['sentiment_class'] = df['global_sentiment_polarity'].apply(
    lambda x: 'positivo' if x > 0 else ('negativo' if x < 0 else 'neutro')
)

sns.boxplot(data=df, x='sentiment_class', y='shares')
plt.title("Shares según tipo de sentimiento global")
plt.show()


🟥 1. OUTLIERS (Caja y bigotes)
📌 Qué aporta al MLOps

Detectar outliers te permite decidir:

si debes capearlos (winsorization),

eliminarlos,

o conservarlos (porque representan viralidad real).

Esto es clave para evitar drift y errores en el pipeline de transformación.

In [ ]:
# 1. Análisis de Outliers por variable
for col in sentiment_vars:
    plt.figure(figsize=(6,3))
    sns.boxplot(x=df[col], color="lightblue")
    plt.title(f"Outliers en {col}")
    plt.show()


🟧 2. SKEWNESS (Asimetría de la distribución)
📌 Qué aporta al MLOps

Las variables muy sesgadas afectan modelos sensibles a escala
(SVR, regresión lineal, MLP).
Aquí decides si necesitas:

MinMaxScaler

RobustScaler

Log-transform

Binning

In [ ]:
# 2. Skewness del Grupo E
skewness = df[sentiment_vars].skew().sort_values(ascending=False)
print("Asimetría (skewness) de las variables del Grupo E:")
skewness


🟨 3. HEATMAP INTERNO DEL GRUPO E
📌 Qué aporta al MLOps

Detecta multicolinealidad.

Permite eliminar variables redundantes.

Reduce dimensionalidad del dataset.

Esto impacta directamente el rendimiento y la estabilidad del modelo.

In [ ]:
# 3. Matriz de correlación interna del Grupo E
plt.figure(figsize=(10,8))
sns.heatmap(df[sentiment_vars].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación — Grupo E (Sentimiento)")
plt.show()


🟩 4. IMPORTANCIA DE VARIABLES (Random Forest)
📌 Qué aporta al MLOps

Aunque la correlación sea baja,
un modelo no lineal como Random Forest puede:

identificar interacciones entre variables,

detectar señales débiles pero útiles,

servir como criterio de selección de features,

alimentar el modelo final del pipeline MLOps.

In [ ]:
# 4. Importancia de variables usando Random Forest
from sklearn.ensemble import RandomForestRegressor

X = df[sentiment_vars]
y = df['shares']

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

model.fit(X, y)

importances = pd.DataFrame({
    'variable': sentiment_vars,
    'importancia': model.feature_importances_
}).sort_values('importancia', ascending=False)

importances


# 🟫 Conclusiones del EDA — Grupo E (Variables de Sentimiento)
*(Con enfoque técnico para MLOps / Feature Engineering / Drift Monitoring)*

---

## 🧩 1. Principales descubrimientos del análisis descriptivo

- Las variables de sentimiento muestran valores centrados y sin rangos extremos, lo cual confirma que el contenido de Mashable tiende a ser moderadamente emocional, sin exageraciones masivas.
- La subjetividad global del contenido tiene una media de **0.44**, indicando un tono relativamente equilibrado entre informativo y opinativo.
- Las polaridades promedio (`global_sentiment_polarity` y `avg_positive_polarity`) presentan distribuciones centradas y estrechas, lo cual implica **poca variabilidad emocional interna** en la mayoría de los artículos.

✔ **Interpretación para ML:**  
Estas variables son estables, poco ruidosas y aportan pequeñas señales complementarias.

---

## ⚖️ 2. Correlación con viralidad (shares): señales débiles pero reales

Aunque la correlación con shares es baja (0.01–0.05), variables como:

- `global_subjectivity` (0.048)  
- `abs_title_sentiment_polarity` (0.039)  
- `title_subjectivity` (0.037)  

indican que **la subjetividad y la emoción en los títulos sí tienen impacto**, especialmente en contenido altamente viral.

✔ **Conclusión:**  
No son variables fuertes individualmente, pero aportan valor marginal que **modelos no lineales** pueden aprovechar mejor que modelos lineales.

---

## 📉 3. Distribuciones sesgadas (skewness): impacto directo en el pipeline

Se detectaron variables con fuerte sesgo (**skew > 1**):

- `global_rate_negative_words` (4.61)  
- `abs_title_sentiment_polarity` (1.75)  
- `global_sentiment_polarity` (0.85)  
- `global_rate_positive_words` (0.84)  

✔ **Implicación MLOps:**  
Estas distribuciones requieren:

- **MinMaxScaler** para modelos sensibles a escala  
- **RobustScaler** para variables con outliers  
- **NO log-transform**, ya que no son conteos ni variables adecuadas para logaritmos

---

## 🚨 4. Outliers detectados: conservarlos por razones de negocio

Las variables de polaridad y subjetividad presentan outliers, pero **NO deben eliminarse** porque:

- representan artículos con contenido emocional atípico,  
- que suelen corresponder a **casos de viralidad extrema**.

✔ **Regla del pipeline:**  
➡️ No eliminar outliers  
➡️ Solo escalar con MinMaxScaler o RobustScaler  

---

## 🔥 5. Heatmap interno: baja multicolinealidad

El heatmap mostró que:

- Las variables del Grupo E tienen **correlaciones internas bajas**.  
- Esto indica que cada una aporta **información distinta**, no redundante.

✔ **Implicación:**  
➡️ Se recomienda **mantener las 10 variables** del Grupo E en el set final de features.

---

## 🌲 6. Importancia del modelo (Random Forest): señales más fuertes que la correlación

Aunque la correlación lineal era muy baja, el modelo Random Forest reveló que estas variables **sí tienen peso predictivo**:

### 🔝 Top 6 variables según RF
1. `global_subjectivity`  
2. `avg_positive_polarity`  
3. `global_rate_positive_words`  
4. `global_sentiment_polarity`  
5. `avg_negative_polarity`  
6. `global_rate_negative_words`  

✔ **Conclusión clave:**  
➡️ Los modelos no lineales sí capturan patrones complejos entre sentimiento y viralidad.  
➡️ Estas variables deben permanecer en el pipeline de entrenamiento.

---

# 🚀 Conclusiones generales del Grupo E

- El sentimiento no explica directamente la viralidad, pero actúa como **señal secundaria útil**.  
- Los títulos subjetivos o emocionalmente intensos aportan señales relevantes para detectar viralidad.  
- Las distribuciones sesgadas requieren **escalamiento robusto** para evitar problemas en el entrenamiento.  
- Los outliers deben conservarse, ya que representan casos reales de viralidad.  
- Los modelos no lineales aprovechan estas variables mejor que los modelos lineales.  
- No existe multicolinealidad fuerte → **todas las variables pueden mantenerse**.

---

# 🧠 7. Recomendaciones para los siguientes pasos del MLOps

---

## 🟦 A. Feature Engineering

Incluir estas variables en el pipeline con:

- **MinMaxScaler** (recomendación principal)  
- **RobustScaler** para las variables:  
  - `global_rate_negative_words`  
  - `global_rate_positive_words`  
  - `abs_title_sentiment_polarity`  
- No aplicar log-transform  
- No eliminar outliers  
- Mantener las 10 variables del Grupo E  

---

## 🟥 B. Modelos recomendados

Estas variables funcionan mejor con:

- **Random Forest**  
- **XGBoost**  
- **LightGBM**  
- **MLPRegressor**  

⚠️ **Evitar modelos lineales**, ya que no capturan bien las relaciones no lineales de este grupo.

---

## 🟧 C. Monitoreo en producción (Drift Monitoring)

Las variables de sentimiento son sensibles a cambios editoriales.  
Monitorear con **EvidentlyAI**:

- `global_sentiment_polarity`  
- `title_sentiment_polarity`  
- `abs_title_sentiment_polarity`  
- `global_rate_negative_words`  

Cambios en el estilo editorial de Mashable podrían afectar la calidad del modelo.

---

## 🟩 D. Documentación para MLflow

Guardar como metadata:

- Las 10 columnas del Grupo E  
- El tipo de escalamiento aplicado  
- Los valores de skewness  
- Los resultados de Random Forest Feature Importance  

Para garantizar reproducibilidad del pipeline.

---

## ✅ Resumen final en una frase

**El Grupo E aporta señales débiles pero útiles; deben mantenerse en el pipeline con escalamiento adecuado y monitoreo de drift, ya que los modelos no lineales sí logran extraer su valor predictivo.**



## 🧩 ¿Por qué NO aplicamos PCA en el Grupo E (Variables de Sentimiento)?

Después del análisis del Grupo E —que incluye distribución, correlaciones internas, outliers, skewness y relevancia con Random Forest— concluimos que **no es necesario ni recomendable aplicar PCA en esta sección del EDA**.

### ✔ Razones técnicas:

1. **No hay multicolinealidad**  
   El heatmap interno mostró correlaciones muy bajas entre las variables de sentimiento.  
   PCA solo es útil cuando existen correlaciones fuertes que justifiquen la reducción de dimensionalidad.

2. **Las variables tienen significado semántico claro**  
   Usar PCA destruiría la interpretabilidad, combinando subjetividad, polaridad y emociones en componentes abstractos.  
   Para análisis de texto y sentimiento, mantener interpretabilidad es crítico.

3. **Son pocas variables (10 en total)**  
   PCA se usa cuando existen decenas o cientos de columnas.  
   Aquí no hay un problema de dimensionalidad.

4. **Los modelos objetivo (RF, XGBoost, LightGBM)**  
   No requieren PCA, manejan:
   - skewness  
   - no normalizan  
   - toleran ruido  
   - capturan relaciones no lineales  

5. **Los modelos lineales no son adecuados para este grupo**  
   Como no usarás modelos lineales como baseline final, PCA tampoco es necesario como paso previo.

---

### 📌 Conclusión

**No aplicamos PCA en el Grupo E porque no aporta beneficio, afecta interpretabilidad y no existe un problema de redundancia o alta dimensionalidad.**  
Se recomienda mantener las 10 variables en bruto y aplicar únicamente escalamiento (MinMaxScaler o RobustScaler) dentro del pipeline.

